# Skin lesion model

Reads a photograph of a mole or skin patch and decides whether it needs a
dermatologist. Trained on HAM10000: 10,015 dermatoscopic images, each labelled
with a diagnosis confirmed by histopathology, follow-up, expert consensus or
confocal microscopy.

**Why handcrafted features and not a CNN.** A dermatologist assessing a lesion
looks at asymmetry, border, colour and diameter — the ABCD rule taught for
decades. A model built on those same measurements can be shown to a clinician
and argued with. A convolutional network would score higher and could not be
interrogated the same way. For a tool that must be reviewed before it is
trusted, that trade is worth making.

In [ ]:
import subprocess, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
print("project root:", ROOT)

## 1. Fetch the data

About 2.8 GB from Harvard Dataverse. Cached, so this is cheap on a re-run.
Licence is CC BY-NC 4.0 — non-commercial — which is why the images stay out of
the repository.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "ml" / "fetch_skin_data.py")], check=True)

In [ ]:
import pandas as pd

meta = pd.read_csv(ROOT / "data" / "real" / "skin" / "metadata.csv")
print(f"{len(meta):,} labelled lesions")
display(meta.groupby(["risk", "diagnosis_name"]).size())

Note how unbalanced this is. Ordinary moles outnumber melanoma roughly six to
one, which is exactly the trap: a model that always says "ordinary mole" scores
67% accuracy and is useless.

## 2. What is measured

32 numbers per image, all of them things a dermatologist would name.

In [ ]:
from app.ai.skin_features import extract_features, FEATURE_NAMES
from PIL import Image

images = ROOT / "data" / "real" / "skin" / "images"
melanoma = meta[meta.diagnosis == "mel"].iloc[0].image_id
mole = meta[meta.diagnosis == "nv"].iloc[0].image_id

for name, label in ((melanoma, "melanoma"), (mole, "ordinary mole")):
    vector = extract_features(Image.open(images / f"{name}.jpg"))
    interesting = ["asymmetry_vertical", "border_irregularity",
                   "colour_clusters", "blue_white_fraction"]
    values = {k: round(float(vector[FEATURE_NAMES.index(k)]), 3)
              for k in interesting}
    print(f"{label:14} {values}")

## 3. Train

In [ ]:
subprocess.run([sys.executable, str(ROOT / "ml" / "train_skin_model.py")], check=True)

In [ ]:
metrics = json.loads((ROOT / "backend/app/ai/artifacts/skin_metrics.json").read_text())
print(json.dumps(metrics, indent=2))

## 4. The number that matters

Macro-F1 across seven classes is around 0.51, which sounds poor. It is the
wrong question.

What a patient needs to know is not *which* of seven diagnoses this is — it is
*does this need a doctor*. Taking the single most likely class misses almost
half of all cancers, because melanoma rarely wins the argmax outright against
a huge population of ordinary moles.

So the referral decision sums the probability across the malignant and
precancerous classes and compares it to a threshold chosen on the validation
split to catch **at least 90%** of them. That deliberately over-refers: roughly
four in ten flagged lesions turn out benign. For a screening tool that is the
right way round.

In [ ]:
from app.ai.skin_service import assess_skin_image

for name, truth in ((melanoma, "melanoma"), (mole, "ordinary mole")):
    payload = (images / f"{name}.jpg").read_bytes()
    result = assess_skin_image(payload, age=58)
    print(f"{truth:14} -> {result['band']:16} concern={result['concern_score']:.3f}")
    print(f"                  {result['advice']}")

## Limitation

HAM10000 is dermatoscopic — taken through a lens pressed against the skin,
under even lighting. A photograph taken on a phone in a village will not look
like that, and the model has never seen one. It is also overwhelmingly
light-skinned European data, which matters a great deal for Bangladesh.

The interface therefore never names a cancer to a patient. It says whether to
see a dermatologist, and it carries a disclaimer in both languages saying a
photograph cannot replace an examination.